# Preprocessing Dataset Mahasiswa S1 Informatika UKDW Tahun Angkatan 2020-2021
**Pipeline:** Load Data → Seleksi Atribut → Cleaning → Transformasi (Derived Features) → Labeling Risiko → Simpan

## 1. Load Data

In [20]:
import pandas as pd
import numpy as np

df = pd.read_excel('dataset/raw/data_mhs_2020_2021_new.xls', engine='xlrd', dtype={'nim': str})

print(f'Jumlah data: {df.shape[0]} baris, {df.shape[1]} kolom')
print('\nKolom tersedia:')
for col in df.columns:
    print(f'  - {col}')
df.head()

Jumlah data: 55 baris, 14 kolom

Kolom tersedia:
  - nim
  - jumlah_semester
  - ipk
  - ips
  - total_sks_lulus
  - jumlah_mk_diulang
  - semester_terakhir_aktif
  - semester_tidak_aktif_berturut
  - jml_mk_wajib
  - jml_mk_pilihan_bebas_non_bidang
  - jml_mk_pilihan_bebas_prodi
  - jml_mk_pilihan_wajib_profil
  - stmhs
  - status


,nim,jumlah_semester,ipk,ips,total_sks_lulus,jumlah_mk_diulang,semester_terakhir_aktif,semester_tidak_aktif_berturut,jml_mk_wajib,jml_mk_pilihan_bebas_non_bidang,jml_mk_pilihan_bebas_prodi,jml_mk_pilihan_wajib_profil,stmhs,status
0,1,11,3.03,NaN,147,6,20251,0,35,2,9,7,AR,A
1,2,11,2.93,NaN,134,5,20251,0,36,0,11,6,AR,A
2,3,7,2.36,NaN,106,5,20231,4,32,2,3,6,AR,T
3,4,11,2.33,2.27,94,17,20251,0,31,1,3,5,AR,A
4,5,11,2.52,NaN,138,10,20251,0,34,0,4,12,AR,A


## 2. Seleksi Atribut

Dari 14 kolom yang tersedia, dipilih atribut yang relevan dengan rule kategorisasi risiko. Kolom yang tidak digunakan:
- `nim` → hanya identifier, tidak relevan untuk inferensi
- `ips` → 19 nilai kosong (34%), tidak reliable; IPK sudah cukup representatif
- `total_sks_lulus` → redundan, sudah tercakup dalam turunan `total_mk`
- `jumlah_mk_diulang` → tidak digunakan dalam rule kategorisasi risiko
- `semester_terakhir_aktif` → tidak digunakan dalam rule kategorisasi risiko
- `stmhs` → seluruh data bernilai 'AR', tidak memberikan variasi informasi

In [21]:
atribut_terpilih = [
    'jumlah_semester',
    'ipk',
    'semester_tidak_aktif_berturut',
    'jml_mk_wajib',
    'jml_mk_pilihan_bebas_non_bidang',
    'jml_mk_pilihan_bebas_prodi',
    'jml_mk_pilihan_wajib_profil',
    'status' 
]

df = df[atribut_terpilih].copy()
print(f'Atribut setelah seleksi: {df.shape[1]} kolom')
df.info()

Atribut setelah seleksi: 8 kolom
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 8 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   jumlah_semester                  55 non-null     int64  
 1   ipk                              55 non-null     float64
 2   semester_tidak_aktif_berturut    55 non-null     int64  
 3   jml_mk_wajib                     55 non-null     int64  
 4   jml_mk_pilihan_bebas_non_bidang  55 non-null     int64  
 5   jml_mk_pilihan_bebas_prodi       55 non-null     int64  
 6   jml_mk_pilihan_wajib_profil      55 non-null     int64  
 7   status                           55 non-null     object 
dtypes: float64(1), int64(6), object(1)
memory usage: 3.6+ KB


## 3. Pembersihan Data (Cleaning)

Pengecekan dan penanganan:
- Identifikasi missing values
- Deteksi duplikasi
- Validasi logika

In [22]:
print('=== Missing Values ===')
print(df.isnull().sum())

print(f'\n=== Duplikat ===')
print(f'Jumlah baris duplikat: {df.duplicated().sum()}')

print(f'\n=== Statistik Deskriptif ===')
print(df.describe().round(2))

=== Missing Values ===
jumlah_semester                    0
ipk                                0
semester_tidak_aktif_berturut      0
jml_mk_wajib                       0
jml_mk_pilihan_bebas_non_bidang    0
jml_mk_pilihan_bebas_prodi         0
jml_mk_pilihan_wajib_profil        0
status                             0
dtype: int64

=== Duplikat ===
Jumlah baris duplikat: 0

=== Statistik Deskriptif ===
       jumlah_semester    ipk  semester_tidak_aktif_berturut  jml_mk_wajib  \
count            55.00  55.00                          55.00         55.00   
mean              9.55   2.89                           0.16         34.85   
std               0.98   0.49                           0.66          1.48   
min               7.00   1.55                           0.00         30.00   
25%               9.00   2.54                           0.00         34.00   
50%               9.00   2.96                           0.00         35.00   
75%              10.50   3.29                    

In [23]:
print('=== Validasi Nilai Logis ===')
print(f'Semester di luar rentang 7-14 : {((df["jumlah_semester"] < 7) | (df["jumlah_semester"] > 14)).sum()} mahasiswa')
print(f'IPK di luar rentang 0-4       : {((df["ipk"] < 0) | (df["ipk"] > 4)).sum()} mahasiswa')
print(f'MK Wajib > 36 (melebihi kurikulum): {(df["jml_mk_wajib"] > 36).sum()} mahasiswa')
print(f'MK Non-Bidang > 3 (melebihi batas): {(df["jml_mk_pilihan_bebas_non_bidang"] > 3).sum()} mahasiswa')

print('\n→ Dataset bersih: tidak ada missing values, duplikat, maupun nilai tidak wajar pada atribut terpilih.')

=== Validasi Nilai Logis ===
Semester di luar rentang 7-14 : 0 mahasiswa
IPK di luar rentang 0-4       : 0 mahasiswa
MK Wajib > 36 (melebihi kurikulum): 0 mahasiswa
MK Non-Bidang > 3 (melebihi batas): 0 mahasiswa

→ Dataset bersih: tidak ada missing values, duplikat, maupun nilai tidak wajar pada atribut terpilih.


## 4. Transformasi Data (Derived Features)

Berdasarkan syarat kelulusan kurikulum, dibuat dua atribut turunan yang akan digunakan langsung dalam rule forward chaining:

| Atribut Turunan | Formula | Kegunaan |
|---|---|---|
| `total_mk_pilihan` | `wajib_profil + bebas_prodi + non_bidang` | Total MK pilihan yang telah diselesaikan |
| `total_mk` | `jml_mk_wajib + total_mk_pilihan` | Total seluruh MK (acuan syarat lulus ≥ 53) |


In [24]:
df['total_mk_pilihan'] = (
    df['jml_mk_pilihan_wajib_profil'] +
    df['jml_mk_pilihan_bebas_prodi'] +
    df['jml_mk_pilihan_bebas_non_bidang']
)

df['total_mk'] = df['jml_mk_wajib'] + df['total_mk_pilihan']

print('=== Distribusi Total MK Pilihan ===')
print(df['total_mk_pilihan'].describe().round(2))

print('\n=== Distribusi Total MK (Wajib + Pilihan) ===')
print(df['total_mk'].describe().round(2))

print(f'\nMahasiswa yang sudah memenuhi total MK ≥ 53 : {(df["total_mk"] >= 53).sum()}')
print(f'Mahasiswa yang belum memenuhi total MK ≥ 53 : {(df["total_mk"] < 53).sum()}')

=== Distribusi Total MK Pilihan ===
count    55.00
mean     15.47
std       3.65
min       2.00
25%      14.00
50%      16.00
75%      18.00
max      21.00
Name: total_mk_pilihan, dtype: float64

=== Distribusi Total MK (Wajib + Pilihan) ===
count    55.00
mean     50.33
std       4.94
min      32.00
25%      49.00
50%      52.00
75%      53.00
max      57.00
Name: total_mk, dtype: float64

Mahasiswa yang sudah memenuhi total MK ≥ 53 : 22
Mahasiswa yang belum memenuhi total MK ≥ 53 : 33


## 5. Labeling Risiko

Setiap mahasiswa diberi label risiko berdasarkan rule forward chaining yang telah disepakati. Sistem memeriksa kondisi dari risiko tertinggi terlebih dahulu (Tinggi → Sedang → Rendah). Mahasiswa dikategorikan ke suatu level risiko jika memenuhi satu atau lebih kondisi pada level tersebut.

**Catatan:** Kondisi yang memerlukan data di luar dataset (peringatan akademik, status keuangan, persoalan keluarga) akan diinput secara manual oleh pengguna saat sistem berjalan dan tidak dimasukkan dalam proses labeling otomatis ini.

### Rule Kategorisasi:

**Risiko Tinggi** jika memenuhi salah satu:
- IPK < 2.00
- Semester pada rentang 13-14
- Tidak aktif > 2 semester berturut-turut
- MK Wajib < 33
- Total MK < 50

**Risiko Sedang** jika memenuhi salah satu:
- IPK pada rentang 2.00-2.75
- Semester pada rentang 10-12
- Tidak aktif = 2 semester berturut-turut
- MK Wajib < 34
- Total MK < 52

**Risiko Rendah** jika memenuhi salah satu:
- IPK > 2.75
- Semester pada rentang 7-9
- Tidak aktif < 2 semester berturut-turut
- MK Wajib ≥ 35
- Total MK ≥ 52

In [25]:
def label_risiko(row):
    ipk = row['ipk']
    smt = row['jumlah_semester']
    tidak_aktif = row['semester_tidak_aktif_berturut']
    mk_wajib = row['jml_mk_wajib']
    total_mk = row['total_mk']

    # --- RISIKO TINGGI ---
    if (
        ipk < 2.00 or
        (13 <= smt <= 14) or
        tidak_aktif > 2 or
        mk_wajib < 33 or
        total_mk < 50
    ):
        return 'Tinggi'

    # --- RISIKO SEDANG ---
    if (
        (2.00 <= ipk <= 2.75) or
        (10 <= smt <= 12) or
        tidak_aktif == 2 or
        mk_wajib < 34 or
        total_mk < 52
    ):
        return 'Sedang'

    # --- RISIKO RENDAH ---
    return 'Rendah'


df['risiko'] = df.apply(label_risiko, axis=1)

print('=== Distribusi Label Risiko ===')
print(df['risiko'].value_counts())
print(f'\nTotal mahasiswa: {len(df)}')

=== Distribusi Label Risiko ===
risiko
Sedang    20
Rendah    18
Tinggi    17
Name: count, dtype: int64

Total mahasiswa: 55


In [26]:
print('=== Crosscheck Risiko vs Status Keaktifan ===')
print(pd.crosstab(df['risiko'], df['status'], margins=True))
print('Keterangan: A = Aktif, T = Tidak Aktif')

print('\n=== Detail Mahasiswa Status T (Tidak Aktif) ===')
cols_tampil = ['jumlah_semester', 'ipk', 'semester_tidak_aktif_berturut',
               'jml_mk_wajib', 'total_mk_pilihan', 'total_mk', 'status', 'risiko']
print(df[df['status'] == 'T'][cols_tampil].to_string())

=== Crosscheck Risiko vs Status Keaktifan ===
status   A  T  All
risiko            
Rendah  17  1   18
Sedang  19  1   20
Tinggi  15  2   17
All     51  4   55
Keterangan: A = Aktif, T = Tidak Aktif

=== Detail Mahasiswa Status T (Tidak Aktif) ===
    jumlah_semester   ipk  semester_tidak_aktif_berturut  jml_mk_wajib  total_mk_pilihan  total_mk status  risiko
2                 7  2.36                              4            32                11        43      T  Tinggi
12               10  1.55                              2            33                 5        38      T  Tinggi
17                9  3.10                              2            36                16        52      T  Sedang
44                8  3.17                              1            36                19        55      T  Rendah


In [27]:
print('=== Dataset Final ===')
print(f'Jumlah baris : {df.shape[0]}')
print(f'Jumlah kolom : {df.shape[1]}')
print(f'Kolom        : {df.columns.tolist()}')
df.head(10)

=== Dataset Final ===
Jumlah baris : 55
Jumlah kolom : 11
Kolom        : ['jumlah_semester', 'ipk', 'semester_tidak_aktif_berturut', 'jml_mk_wajib', 'jml_mk_pilihan_bebas_non_bidang', 'jml_mk_pilihan_bebas_prodi', 'jml_mk_pilihan_wajib_profil', 'status', 'total_mk_pilihan', 'total_mk', 'risiko']


,jumlah_semester,ipk,semester_tidak_aktif_berturut,jml_mk_wajib,jml_mk_pilihan_bebas_non_bidang,jml_mk_pilihan_bebas_prodi,jml_mk_pilihan_wajib_profil,status,total_mk_pilihan,total_mk,risiko
0,11,3.03,0,35,2,9,7,A,18,53,Sedang
1,11,2.93,0,36,0,11,6,A,17,53,Sedang
2,7,2.36,4,32,2,3,6,T,11,43,Tinggi
3,11,2.33,0,31,1,3,5,A,9,40,Tinggi
4,11,2.52,0,34,0,4,12,A,16,50,Sedang
5,11,2.07,0,35,0,8,11,A,19,54,Sedang
6,11,2.64,0,34,1,7,8,A,16,50,Sedang
7,11,2.96,0,35,2,5,9,A,16,51,Sedang
8,10,2.64,0,33,2,7,5,A,14,47,Tinggi
9,11,2.54,0,36,2,7,12,A,21,57,Sedang


## 6. Simpan Dataset

In [28]:
df.to_csv('dataset/processed/data_mhs_2020_2021_processed.csv', index=False)
print(f'Dataset berhasil disimpan: {df.shape[0]} baris, {df.shape[1]} kolom')
print(f'Kolom: {df.columns.tolist()}')

Dataset berhasil disimpan: 55 baris, 11 kolom
Kolom: ['jumlah_semester', 'ipk', 'semester_tidak_aktif_berturut', 'jml_mk_wajib', 'jml_mk_pilihan_bebas_non_bidang', 'jml_mk_pilihan_bebas_prodi', 'jml_mk_pilihan_wajib_profil', 'status', 'total_mk_pilihan', 'total_mk', 'risiko']
